# S2 | Question Schema & Semantic Groupings

Reference notebook: documents the full SEM_SCHEMA taxonomy used to annotate the 1k VQA pool,
analyses each field's distribution, and explains the grouping decisions used across S2 notebooks.

**Key paths**
- Schema definition: `api/prompts.py` — `SEM_SCHEMA`, `SYSTEM_SEM`, `CTL_SCHEMA`
- Semantics data: `dataset/vqa/vqa1k_semantics.jsonl` (1000 entries)
- Control variants: `dataset/vqa/vqa1k_control.jsonl`
- S1 question type notebook: `analysis/session1/s1_question_type.ipynb`
- S2 sample: `experiment/s2.csv` (100 questions)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
from pathlib import Path

BASE = Path('/home/david/Desktop/yuna/HPA')
SEM_PATH  = BASE / 'dataset/vqa/vqa1k_semantics.jsonl'
CTRL_PATH = BASE / 'dataset/vqa/vqa1k_control.jsonl'
S2_CSV    = BASE / 'experiment/s2.csv'
S1_CSV    = BASE / 'analysis/session1/csv/human_vqa.csv'

sem = pd.DataFrame([json.loads(l) for l in open(SEM_PATH)])
ctrl = {int(json.loads(l)['question_id']): json.loads(l) for l in open(CTRL_PATH)}
s2  = pd.read_csv(S2_CSV)
s1  = pd.read_csv(S1_CSV)

# Merge semantics into s2
s2 = s2.merge(sem[['question_id','w','op','ent','attr','ans','space','know','txt']], on='question_id', how='left')

print(f'1k pool: {len(sem)} questions')
print(f'S1 sample: {len(s1)} questions')
print(f'S2 sample: {len(s2)} questions')

---

## SEM_SCHEMA Field Reference

Defined in `api/prompts.py:72`. Each question in the 1k pool is annotated with:

| Field | Type | Values | Meaning |
|-------|------|--------|---------|
| `w` | str | what, which, who, where, when, why, how, how_many, how_much, how_long, how_old, yesno, other | Question word |
| `op` | str | exist, ident, attr, count, act, spat, temp, text, know, cause, comp, other | Operator / reasoning type |
| `ent` | str | person, animal, vehicle, food, product, place, text, object, other | Primary entity type |
| `attr` | str\|null | color, age, count, dur, mat, ident, type, name, loc, other | Attribute being asked |
| `ans` | str | bool, num, dur, age, color, mat, loc, person, object, cat, text, open | Expected answer type |
| `space` | str | bin, num, cat, open | Answer space |
| `sub` | str | — | Subject noun phrase |
| `obj` | str\|null | — | Object noun phrase |
| `p` | str\|null | — | Predicate |
| `sp` | list | left, right, top, bottom, front, back, center, foreground, background | Spatial markers |
| `rel` | list | on, in, next, behind, hold, wear, under, above, between, near | Relations |
| `dx` | list | this, that, these, those, here, there | Deictic markers |
| `neg` | bool | — | Negation present |
| `know` | bool | — | Requires world knowledge |
| `txt` | bool | — | Requires reading text in image |

**op priority rule** (from `SYSTEM_SEM`): text > count > spat > temp > cause > comp > attr > exist > ident > act > know > other

In [ ]:
# ── Per-field value distributions ───────────────────────────────────────────
fields = ['w', 'op', 'ent', 'attr', 'ans', 'space']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

COLORS = {'w':'#3498db', 'op':'#e74c3c', 'ent':'#27ae60',
          'attr':'#f39c12', 'ans':'#9b59b6', 'space':'#1abc9c'}

for ax, field in zip(axes.flat, fields):
    counts = sem[field].value_counts()
    bars = ax.barh(counts.index, counts.values, color=COLORS[field], alpha=0.8, edgecolor='white')
    for bar, v in zip(bars, counts.values):
        ax.text(v + 3, bar.get_y() + bar.get_height()/2, str(v), va='center', fontsize=8)
    ax.set_xlabel('Count'); ax.set_xlim(0, counts.max() * 1.18)
    ax.set_title(f'`{field}` ({len(counts)} values)', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('SEM_SCHEMA Field Distributions — 1k VQA Pool', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE / 'analysis/figures/s2_schema_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---

## `op` — Operator Types

The operator captures *what reasoning type* the question requires:

| op | Count | Description | Typical question |
|----|-------|-------------|------------------|
| `attr` | 428 | Attribute lookup (color, type, material, name) | *What color is the car?* |
| `ident` | 109 | Entity identification | *What is the man holding?* |
| `count` | 98 | Counting | *How many dogs are there?* |
| `exist` | 95 | Existential presence | *Is there a cat?* |
| `spat` | 81 | Spatial location/relation | *Where is the phone?* |
| `act` | 78 | Activity/action | *What is the woman doing?* |
| `text` | 36 | Reading text in image | *What does the sign say?* |
| `know` | 23 | World knowledge beyond visual | *What country are they in?* |
| `comp` | 18 | Comparative/superlative | *Which is taller?* |
| `cause` | 14 | Causal reasoning | *Why is the man running?* |
| `temp` | 10 | Temporal / duration | *How old is the child?* |
| `other` | 10 | Unclassified | — |

**Grouping used in S2 analysis** (2-way):
- `attr` → **attr** (largest distinct group; color/material/type biases well-studied)
- everything else → **other_op** (collapsed to keep 5×2 grid tractable)

---

## `ent` — Entity Types

The entity captures *what the primary referent is*:

| ent | Count | Description | Typical question |
|-----|-------|-------------|------------------|
| `object` | 344 | Generic physical object | *What color is the chair?* |
| `person` | 232 | Human | *What is the man wearing?* |
| `animal` | 107 | Non-human animal | *What color is the dog?* |
| `other` | 84 | Abstract / situational | *What sport is being played?* |
| `food` | 78 | Food & drink | *What is in the bowl?* |
| `place` | 60 | Room / location / scene | *What room is this?* |
| `vehicle` | 52 | Vehicles | *What color is the bus?* |
| `text` | 26 | Written text as entity | *What does the sign say?* |
| `product` | 17 | Branded / manufactured item | *What brand of shoes?* |

**Grouping used in S2 analysis** (5-way):

| Group | Entities | Pool n | Rationale |
|-------|----------|--------|-----------|
| `object` | object | 344 | Largest generic category; broad object priors |
| `person` | person | 232 | Human-specific priors (clothing, activity, gender) |
| `animal` | animal | 107 | Kept distinct: strong species/color priors |
| `scene` | food + place + vehicle | 190 | Shared: scene-level context priors |
| `misc` | other + text + product | 118 | Abstract/situational; no strong visual anchor |

In [ ]:
# ── Cross-tab heatmaps: op × ent, op × ans, op × space ─────────────────────
def plot_crosstab(ax, df, row_col, col_col, title, cmap='Blues'):
    ct = pd.crosstab(df[row_col], df[col_col])
    im = ax.imshow(ct.values, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(ct.columns))); ax.set_xticklabels(ct.columns, rotation=35, ha='right', fontsize=8)
    ax.set_yticks(range(len(ct.index)));   ax.set_yticklabels(ct.index, fontsize=8)
    ax.set_title(title, fontweight='bold', fontsize=10)
    for i in range(len(ct.index)):
        for j in range(len(ct.columns)):
            v = ct.values[i, j]
            if v > 0:
                ax.text(j, i, str(v), ha='center', va='center', fontsize=7,
                        color='white' if v > ct.values.max()*0.5 else 'black')
    return im

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
plot_crosstab(axes[0], sem, 'op', 'ent',   'op × ent')
plot_crosstab(axes[1], sem, 'op', 'ans',   'op × ans',   cmap='Greens')
plot_crosstab(axes[2], sem, 'op', 'space', 'op × space', cmap='Oranges')
plt.suptitle('Cross-tab Heatmaps — 1k VQA Pool', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE / 'analysis/figures/s2_schema_crosstabs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Question word × op cross-tab ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ct = pd.crosstab(sem['w'], sem['op'])
im = ax.imshow(ct.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(ct.columns))); ax.set_xticklabels(ct.columns, rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(ct.index)));   ax.set_yticklabels(ct.index, fontsize=9)
ax.set_title('Question Word × Operator (w × op) — 1k VQA Pool', fontweight='bold', fontsize=12)
for i in range(len(ct.index)):
    for j in range(len(ct.columns)):
        v = ct.values[i, j]
        if v > 0:
            ax.text(j, i, str(v), ha='center', va='center', fontsize=8,
                    color='white' if v > ct.values.max()*0.5 else 'black')
plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.savefig(BASE / 'analysis/figures/s2_schema_w_op.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key w→op mappings (deterministic from schema):')
print('  how_many → count (100%)')
print('  where    → spat  (97%)')
print('  why      → cause (86%)')
print('  who      → ident (86%)')
print('  what     → attr/ident/act/spat (multi-way split)')
print('  yesno    → exist/attr/ident/act/... (multi-way, broad)')

In [ ]:
# ── Structural flags: sp, rel, dx, neg, know, txt ──────────────────────────
sp_flat  = [v for lst in sem['sp']  for v in (lst if isinstance(lst, list) else [])]
rel_flat = [v for lst in sem['rel'] for v in (lst if isinstance(lst, list) else [])]
dx_flat  = [v for lst in sem['dx']  for v in (lst if isinstance(lst, list) else [])]

print(f'Questions with spatial markers (sp):  {(sem["sp"].apply(len) > 0).sum()} / {len(sem)}')
print(f'  values: {Counter(sp_flat).most_common()}')
print(f'Questions with relations (rel):       {(sem["rel"].apply(len) > 0).sum()} / {len(sem)}')
print(f'  values: {Counter(rel_flat).most_common()}')
print(f'Questions with deictic (dx):          {(sem["dx"].apply(len) > 0).sum()} / {len(sem)}')
print(f'  values: {Counter(dx_flat).most_common()}')
print(f'Negation (neg=True):                  {sem["neg"].sum()} / {len(sem)}')
print(f'World knowledge (know=True):          {sem["know"].sum()} / {len(sem)}')
print(f'Text reading (txt=True):              {sem["txt"].sum()} / {len(sem)}')

# Note: dx flags are crucial for deictic_removed control type
print(f'\nDeictic questions (dx non-empty) = candidate deictic_removed differences')

---

## S2 Sample Composition

The 100-question S2 sample was drawn from the pool of ~412 non-S1, non-yes/no questions
using a **5×2 stratified equal-quota design** ranked by signal score.
An additional **10 flat questions** were added manually to `s2_pronominalized.json` (IDs 301–310).

See `analysis/session2/01_sampling.ipynb` for full sampling procedure.

In [ ]:
# ── S2 sample: fine-grained op and ent breakdown ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ENT5_MAP = {'object':'object','person':'person','animal':'animal',
            'food':'scene','place':'scene','vehicle':'scene',
            'other':'misc','text':'misc','product':'misc'}
s2['ent_grp'] = s2['ent'].map(ENT5_MAP).fillna('misc')

for ax, (col, title, pool_col) in zip(axes, [
        ('op', 'Operator (op) — fine-grained', 'op'),
        ('ent', 'Entity (ent) — fine-grained', 'ent')]):
    s2_counts   = s2[col].value_counts()
    pool_counts = sem[pool_col].value_counts()
    labels = [l for l in pool_counts.index if l in s2_counts or pool_counts[l] > 5]
    x = np.arange(len(labels)); w = 0.35
    pool_pct = [pool_counts.get(l, 0) / len(sem) * 100 for l in labels]
    s2_pct   = [s2_counts.get(l, 0)   / len(s2)  * 100 for l in labels]
    ax.bar(x - w/2, pool_pct, w, label='1k pool', color='#bdc3c7', alpha=0.85)
    ax.bar(x + w/2, s2_pct,  w, label='S2 (100)', color='#3498db', alpha=0.9)
    for xi, (p, s) in enumerate(zip(pool_pct, s2_pct)):
        ax.text(xi-w/2, p+0.5, f'{pool_counts.get(labels[xi],0)}', ha='center', fontsize=7, color='#555')
        ax.text(xi+w/2, s+0.5, f'{s2_counts.get(labels[xi],0)}',   ha='center', fontsize=7, color='#2980b9')
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('% of questions'); ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('S2 Sample vs 1k Pool — Fine-grained op & ent', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE / 'analysis/figures/s2_schema_sample_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

print('S2 sample op breakdown:')
print(s2['op'].value_counts().to_string())
print('\nS2 sample ent breakdown:')
print(s2['ent'].value_counts().to_string())
print('\nS2 sample ent_grp (5-way):')
print(s2['ent_grp'].value_counts().to_string())

---

## Grouping Decision Summary

| Dimension | Fine-grained (12 values) | S2 grouping | Rationale |
|-----------|--------------------------|-------------|-----------|
| `op` | exist, ident, attr, count, act, spat, temp, text, know, cause, comp, other | **attr** vs **other_op** | `attr` (428) is the dominant type with distinct color/type priors; all others collapsed |
| `ent` | object, person, animal, food, place, vehicle, text, product, other | **object / person / animal / scene / misc** | Keeps the 3 largest distinct; merges scene-level (food+place+vehicle) and abstract (other+text+product) |

### What the groupings capture for shortcut analysis

- **attr × object** — color/material biases for generic objects (largest cell, ~170 pool Qs)
- **attr × person** — clothing/age/gender priors for people
- **other_op × person** — action/identity priors (who, what doing)
- **other_op × animal** — species/color priors (small but clean)
- **scene** — contextual priors: food→'pizza', place→'bedroom', vehicle→'white'
- **misc** — abstract situational: sport→'baseball', weather→'sunny', country→'USA'